In [1]:
# @title 1) 드라이브 마운트 & 경로 설정
from google.colab import drive
drive.mount('/content/drive')

# ===== 경로/파일명 =====
DATA_DIR        = "/content/drive/MyDrive/data/instacart"
MASTER_PATH     = f"{DATA_DIR}/master_dataset_with_roles_final.csv"

# 임베딩 입력 (동일 순서 보장)
ADDRESS_PATH    = f"{DATA_DIR}/address(in).csv"   # user_id 열 필요
EMBED_PATH      = f"{DATA_DIR}/embedding.npy"     # (n_users, 128)

# (이미 보유한) 베이스라인 결과 파일
XGB_ORD_PATH    = f"{DATA_DIR}/test_predictions_orders_threshold_0.32.csv"
XGB_DETAIL_PATH = f"{DATA_DIR}/test_predictions_detailed.csv"  # 있으면 AUC/Logloss 계산

# 임베딩 XGB 산출물
OUT_OOF_PATH    = f"{DATA_DIR}/performance3/oof_predictions_instacart_embedding.csv"
OUT_DET_TEST    = f"{DATA_DIR}/performance3/test_predictions_detailed_embedding.csv"
OUT_ORD_TEST    = f"{DATA_DIR}/performance3/test_predictions_orders_embedding.csv"
OUT_META_PATH   = f"{DATA_DIR}/performance3/embedding_xgb_meta.json"

# 최종 비교 산출물
OUT_SUMMARY_CSV  = f"{DATA_DIR}/performance3/xgb_baseline_vs_embedding_summary.csv"
OUT_SUMMARY_JSON = f"{DATA_DIR}/performance3/xgb_baseline_vs_embedding_summary.json"
OUT_PER_ORDER_CMP= f"{DATA_DIR}/performance3/baseline_vs_embedding_per_order_f1.csv"
OUT_EMB_TOP50    = f"{DATA_DIR}/performance3/embedding_better_top50.csv"
OUT_BASE_TOP50   = f"{DATA_DIR}/performance3/baseline_better_top50.csv"


Mounted at /content/drive


In [2]:
# @title 2) 유틸 함수
import os, json, re, time, subprocess, gc
from dataclasses import dataclass, asdict
from typing import List, Optional, Dict, Tuple

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.feature_selection import VarianceThreshold
import xgboost as xgb

# ----- 공통 유틸 -----
def first_existing(cands: List[str], cols: List[str]) -> Optional[str]:
    for c in cands:
        if c in cols: return c
    return None

def is_binary_series(s: pd.Series) -> bool:
    vals = pd.unique(s.dropna())
    return set(vals.tolist()).issubset({0,1}) or set(vals.tolist()).issubset({0.0,1.0})

def f1_single(true_set: set, pred_set: set) -> float:
    # Instacart 규칙: 예측/정답 모두 빈 집합이면 'None' 매칭으로 F1=1
    if len(true_set) == 0 and len(pred_set) == 0:
        return 1.0
    if len(pred_set) == 0:
        pred_set = {"None"}
    if len(true_set) == 0:
        true_set = {"None"}
    tp = len(true_set & pred_set)
    if tp == 0:
        return 0.0
    precision = tp / len(pred_set)
    recall    = tp / len(true_set)
    return 0.0 if (precision+recall)==0 else 2*precision*recall/(precision+recall)

def parse_products_to_list(s) -> list:
    """문자열에서 product_id 리스트 파싱: '1 2', '1, 2', '[1,2]' 등 모두 허용."""
    if pd.isna(s): return []
    s = str(s).strip()
    if s == "" or s.lower() == "none": return []
    toks = re.findall(r"\d+", s)
    return toks if len(toks) > 0 else [t for t in s.split() if t]

def order_level_f1(df: pd.DataFrame, order_col: str, product_col: str,
                   target_col: str, proba_col: str, thr: float) -> float:
    f1s = []
    for _, g in df.groupby(order_col):
        true_set = set(g.loc[g[target_col]==1, product_col].tolist())
        pred_set = set(g.loc[g[proba_col]>=thr, product_col].tolist())
        f1s.append(f1_single(true_set, pred_set))
    return float(np.mean(f1s)) if len(f1s) else 0.0

def search_best_threshold_fast(
    df: pd.DataFrame, order_col: str, product_col: str,
    target_col: str, proba_col: str,
    quantile_lo: float = 0.05, quantile_hi: float = 0.95,
    n_candidates: int = 31, max_orders: int = 50_000,
    seed: int = 2025, verbose: bool = True
) -> Tuple[float, float]:
    t0 = time.time()
    orders = df[order_col].drop_duplicates()
    if (max_orders is not None) and (len(orders) > max_orders):
        sampled_orders = orders.sample(max_orders, random_state=seed)
        sub = df[df[order_col].isin(sampled_orders)][[order_col, product_col, target_col, proba_col]].copy()
        if verbose: print(f"[thr-search] sample orders: {len(sampled_orders):,}  rows: {len(sub):,}")
    else:
        sub = df[[order_col, product_col, target_col, proba_col]].copy()
        if verbose: print(f"[thr-search] full orders: {sub[order_col].nunique():,}  rows: {len(sub):,}")

    qs = np.linspace(quantile_lo, quantile_hi, n_candidates)
    thr_list = np.unique(sub[proba_col].quantile(qs).values)
    if verbose: print(f"[thr-search] candidates: {len(thr_list)}  (quantiles {quantile_lo:.2f}~{quantile_hi:.2f})")

    true_cnt = sub.groupby(order_col)[target_col].sum().astype(np.int32)

    best_thr, best_f1 = 0.5, -1.0
    for i, t in enumerate(thr_list, 1):
        mask    = (sub[proba_col] >= t)
        pred_cnt= sub.loc[mask].groupby(order_col, observed=True)[proba_col].size()
        tp_cnt  = sub.loc[mask & (sub[target_col] == 1)].groupby(order_col, observed=True)[target_col].size()

        agg = pd.DataFrame({
            'true': true_cnt,
            'pred': pred_cnt.reindex(true_cnt.index, fill_value=0).astype(np.int32),
            'tp'  : tp_cnt.reindex(true_cnt.index,  fill_value=0).astype(np.int32),
        })
        true = agg['true'].values
        pred = agg['pred'].values
        tp   = agg['tp'].values

        none_case = (pred == 0) & (true == 0)
        with np.errstate(divide='ignore', invalid='ignore'):
            precision = np.divide(tp, pred, out=np.zeros_like(tp, dtype=float), where=pred>0)
            recall    = np.divide(tp, true, out=np.zeros_like(tp, dtype=float), where=true>0)
            denom     = precision + recall
            f1_arr    = np.divide(2*precision*recall, denom, out=np.zeros_like(denom), where=denom>0)
        f1_arr[none_case] = 1.0

        f1 = float(f1_arr.mean())
        if verbose and (i % max(1, len(thr_list)//5) == 0 or i == len(thr_list)):
            print(f"[thr-search] {i}/{len(thr_list)}  thr={t:.4f}  F1={f1:.5f}")
        if f1 > best_f1:
            best_f1, best_thr = f1, float(t)

    if verbose: print(f"[thr-search] best_thr={best_thr:.4f}  best_f1={best_f1:.5f}  total={time.time()-t0:.1f}s")
    return best_thr, best_f1

def fast_auc_logloss(df: pd.DataFrame, y_col: str, p_col: str, max_rows: int = 2_000_000, seed: int = 2025):
    y = df[y_col].to_numpy()
    p = np.clip(df[p_col].to_numpy(dtype=np.float64), 1e-15, 1-1e-15)
    if len(y) > max_rows:
        df = df.sample(n=max_rows, random_state=seed)
        y = df[y_col].to_numpy()
        p = np.clip(df[p_col].to_numpy(dtype=np.float64), 1e-15, 1-1e-15)
        print(f"[metrics] sampled {len(y):,} rows for AUC/Logloss")
    auc = roc_auc_score(y, p) if len(np.unique(y))>1 else np.nan
    ll  = float(log_loss(y, p))
    return auc, ll

# xgboost 3.0.4 — Booster API helper
def _bst_predict_proba(bst: xgb.Booster, dmat: xgb.DMatrix) -> np.ndarray:
    bi = getattr(bst, "best_iteration", None)
    if bi is not None:
        try:
            return bst.predict(dmat, iteration_range=(0, int(bi)+1)).astype(np.float32)
        except TypeError:
            pass
        try:  # fallback
            return bst.predict(dmat, ntree_limit=getattr(bst, "best_ntree_limit", int(bi)+1)).astype(np.float32)
        except Exception:
            pass
    return bst.predict(dmat).astype(np.float32)

def detect_device_and_tree_method():
    try:
        _ = subprocess.check_output(["nvidia-smi"])
        return "cuda", "hist"
    except Exception:
        return "cpu", "hist"

DEVICE, TREE_METHOD = detect_device_and_tree_method()
print(f"[INFO] device={DEVICE}  tree_method={TREE_METHOD}  xgboost={xgb.__version__}")


[INFO] device=cuda  tree_method=hist  xgboost=3.0.4


In [3]:
# @title 3) 마스터테이블 로딩 & 6번째 주문 분리(GT 생성)
df = pd.read_csv(MASTER_PATH, low_memory=False)
cols = df.columns.tolist()

target_col  = first_existing(['reordered','label','target','y','is_reordered'], cols)
product_col = first_existing(['product_id','pid','product'], cols)
order_col   = first_existing(['order_id','oid','order'], cols)
member_col  = first_existing(['user_id','member_id','uid','user'], cols)
eval_col    = 'eval_set' if 'eval_set' in cols else None
assert target_col and product_col and (order_col or member_col), "필수 컬럼 누락"

# 7번(test) 제거, 1~5(prior) 학습 / 6(train) 평가
if eval_col:
    ev = df[eval_col].astype(str).str.lower()
    df = df.loc[~ev.eq('test')].copy()
    mask_train = df[eval_col].astype(str).str.lower().eq('prior')  # 1~5
    mask_test  = df[eval_col].astype(str).str.lower().eq('train')  # 6
else:
    mask_train = (df['role_train'].fillna(0).astype(int)==1) if 'role_train' in df.columns else pd.Series([True]*len(df))
    mask_test  = (df['role_test'].fillna(0).astype(int)==1)  if 'role_test'  in df.columns else pd.Series([False]*len(df))

df_train = df.loc[mask_train].copy()
df_test  = df.loc[mask_test].copy()

# 타깃 이진화
if not is_binary_series(df_train[target_col]):
    df_train[target_col] = (df_train[target_col] > 0).astype(np.int8)
if (target_col in df_test.columns) and (not is_binary_series(df_test[target_col])):
    df_test[target_col] = (df_test[target_col] > 0).astype(np.int8)

order_key = order_col if order_col else member_col
group_key = member_col if member_col else order_col

print(f"[INFO] rows — train(prior)={len(df_train):,}, test(6th)={len(df_test):,}")


[INFO] rows — train(prior)=20,641,991, test(6th)=1,384,617


In [4]:
# @title 4) 임베딩 로드 & 피처 구성(임베딩+수치)
# address(in).csv : embedding.npy 행 순서와 동일한 user_id 나열
addr = pd.read_csv(ADDRESS_PATH)
user_ids_in_order = addr['user_id'].astype(str).tolist()
# 중복 제거하며 순서 유지
seen=set(); ordered_unique=[]
for u in user_ids_in_order:
    if u not in seen:
        seen.add(u); ordered_unique.append(u)

emb = np.load(EMBED_PATH)  # (n_users, d)
assert emb.shape[0] == len(ordered_unique), f"임베딩 행수({emb.shape[0]}) != user_id 수({len(ordered_unique)})"
emb_dim = emb.shape[1]
emb_cols = [f"emb_{i}" for i in range(emb_dim)]
df_emb = pd.DataFrame(emb, columns=emb_cols)
df_emb.insert(0, member_col, ordered_unique)
for c in emb_cols: df_emb[c] = df_emb[c].astype('float32')
print(f"[INFO] embedding: users={len(df_emb):,}, dim={emb_dim}")

# 원본 수치 피처(타깃/ID/역할 제외)
exclude = {c for c in [target_col, member_col, product_col, order_col, eval_col, 'role_train','role_test'] if c}
num_cols = [c for c in df.columns if (c not in exclude) and pd.api.types.is_numeric_dtype(df[c])]
# 아주 작은 분산 제거
if len(num_cols):
    vt = VarianceThreshold(threshold=1e-10)
    vt.fit(df_train[num_cols].fillna(-1.0).astype('float32').values)
    keep_idx = np.where(vt.get_support())[0].tolist()
    num_cols = [num_cols[i] for i in keep_idx]
print(f"[INFO] base numeric features: {len(num_cols)} → {num_cols}")

USE_BASE_NUMERIC = True  # 임베딩 + 수치(권장)
base_cols = num_cols if USE_BASE_NUMERIC else []
FEATURES = emb_cols + base_cols
print(f"[INFO] total features: {len(FEATURES)} (emb {len(emb_cols)} + base {len(base_cols)})")

# 캐스팅/결측
for c in base_cols:
    df_train[c] = df_train[c].astype('float32')
    df_test[c]  = df_test[c].astype('float32')
df_train[base_cols] = df_train[base_cols].fillna(-1.0)
df_test[base_cols]  = df_test[base_cols].fillna(-1.0)

# train 샘플링(음성 다운샘플 + 상한)
NEG_POS_RATIO      = 5.0
MAX_TRAIN_ROWS_EMB = 3_000_000
RANDOM_SEED        = 2025
np.random.seed(RANDOM_SEED)

pos_mask = (df_train[target_col] == 1)
neg_mask = ~pos_mask
n_pos = int(pos_mask.sum())
max_neg = int(min(neg_mask.sum(), NEG_POS_RATIO * n_pos))
neg_idx = df_train.loc[neg_mask].sample(n=max_neg, random_state=RANDOM_SEED).index if max_neg>0 else df_train.loc[neg_mask].index
train_idx = df_train.loc[pos_mask].index.union(neg_idx)
if len(train_idx) > MAX_TRAIN_ROWS_EMB:
    train_idx = pd.Index(np.random.choice(train_idx, size=MAX_TRAIN_ROWS_EMB, replace=False))

df_tr_s = df_train.loc[train_idx].copy()
print(f"[INFO] train sample: {len(df_tr_s):,} rows (pos={int((df_tr_s[target_col]==1).sum()):,})")

# 임베딩 조인
df_tr_s[member_col] = df_tr_s[member_col].astype(str)
df_test[member_col]  = df_test[member_col].astype(str)
df_tr_s = df_tr_s.merge(df_emb, on=member_col, how='left', validate='many_to_one')
df_test  = df_test.merge(df_emb, on=member_col, how='left', validate='many_to_one')

miss_tr = df_tr_s[emb_cols].isna().any(axis=1).sum()
miss_te = df_test[emb_cols].isna().any(axis=1).sum()
if miss_tr or miss_te:
    print(f"[WARN] missing embedding — train {miss_tr:,}  test {miss_te:,} — filling 0")
    df_tr_s[emb_cols] = df_tr_s[emb_cols].fillna(0.0).astype('float32')
    df_test[emb_cols] = df_test[emb_cols].fillna(0.0).astype('float32')


[INFO] embedding: users=205,955, dim=128
[INFO] base numeric features: 7 → ['add_to_cart_order', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order', 'aisle_id', 'department_id']
[INFO] total features: 135 (emb 128 + base 7)
[INFO] train sample: 3,000,000 rows (pos=1,768,007)
[WARN] missing embedding — train 11  test 67 — filling 0


In [5]:
# @title 5) XGBoost(3.0.4, Booster API) 학습 — 임베딩 입력
@dataclass
class XGBCfg:
    n_estimators: int = 600
    max_depth: int = 7
    learning_rate: float = 0.05
    subsample: float = 0.90
    colsample_bytree: float = 0.90
    min_child_weight: float = 1.0
    reg_lambda: float = 1.0
    gamma: float = 0.0
    tree_method: str = TREE_METHOD
    device: str = DEVICE
    random_state: int = 42

cfg = XGBCfg()
NFOLDS = 3
EARLY_STOP = 50

def fit_xgb_cv_booster(df_tr: pd.DataFrame, features: List[str], target: str,
                       groups: Optional[pd.Series], cfg: XGBCfg,
                       model_prefix: str, n_splits: int = NFOLDS,
                       early_stopping_rounds: int = EARLY_STOP) -> Dict:
    X = df_tr[features].values
    y = df_tr[target].values.astype(np.float32)
    if groups is None:
        groups = np.arange(len(y)) % n_splits
    splitter = GroupKFold(n_splits=n_splits)

    params = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "max_depth": cfg.max_depth,
        "eta": cfg.learning_rate,
        "subsample": cfg.subsample,
        "colsample_bytree": cfg.colsample_bytree,
        "min_child_weight": cfg.min_child_weight,
        "lambda": cfg.reg_lambda,
        "gamma": cfg.gamma,
        "tree_method": "hist",   # 3.x: hist + device
        "device": cfg.device,
        "verbosity": 1,
        "seed": cfg.random_state,
    }

    oof = np.zeros(len(df_tr), dtype=np.float32)
    models, metrics = [], []
    for fold, (tr_idx, va_idx) in enumerate(splitter.split(X, y, groups=groups), 1):
        dtr = xgb.DMatrix(X[tr_idx], label=y[tr_idx], feature_names=features)
        dva = xgb.DMatrix(X[va_idx], label=y[va_idx], feature_names=features)
        bst = xgb.train(
            params=params, dtrain=dtr, num_boost_round=cfg.n_estimators,
            evals=[(dva, "valid")], early_stopping_rounds=early_stopping_rounds,
            verbose_eval=False
        )
        proba_va = _bst_predict_proba(bst, dva)
        oof[va_idx] = proba_va
        auc = roc_auc_score(y[va_idx], proba_va) if len(np.unique(y[va_idx]))>1 else np.nan
        ll  = float(log_loss(y[va_idx], np.clip(proba_va,1e-15,1-1e-15)))
        print(f"[EMB] Fold {fold} | AUC={auc:.5f} | Logloss={ll:.5f} | BestIter={getattr(bst,'best_iteration',None)}")
        bst.save_model(os.path.join(DATA_DIR, f"{model_prefix}_fold{fold}.json"))
        models.append(bst); metrics.append({"fold":fold,"auc":float(auc),"logloss":float(ll)})
        del dtr, dva; gc.collect()
    return {"oof":oof, "models":models, "metrics":metrics, "features":features}

groups = df_tr_s[group_key] if group_key in df_tr_s.columns else None
res_EMB = fit_xgb_cv_booster(df_tr_s, FEATURES, target_col, groups, cfg, "xgb_EMB")

df_tr_s["proba_emb"] = res_EMB["oof"]

# OOF 임계치 탐색 + OOF AUC/Logloss
thr_emb, f1_emb = search_best_threshold_fast(
    df_tr_s[[order_key, product_col, target_col, "proba_emb"]],
    order_key, product_col, target_col, "proba_emb",
    n_candidates=31, max_orders=50_000, verbose=True
)
auc_emb, ll_emb = fast_auc_logloss(df_tr_s, target_col, "proba_emb", max_rows=2_000_000)
print(f"[OOF-EMB] BestThr={thr_emb:.4f} | F1={f1_emb:.5f} | AUC≈{auc_emb:.5f} | Logloss≈{ll_emb:.5f}")

# OOF 저장
df_tr_s[[order_key, member_col, product_col, target_col, "proba_emb"]].to_csv(OUT_OOF_PATH, index=False)
print("Saved OOF:", OUT_OOF_PATH)


[EMB] Fold 1 | AUC=0.81344 | Logloss=0.50367 | BestIter=597
[EMB] Fold 2 | AUC=0.81323 | Logloss=0.50347 | BestIter=597
[EMB] Fold 3 | AUC=0.81443 | Logloss=0.50186 | BestIter=599


/tmp/ipython-input-2669753374.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_tr_s["proba_emb"] = res_EMB["oof"]


[thr-search] sample orders: 50,000  rows: 107,930
[thr-search] candidates: 31  (quantiles 0.05~0.95)
[thr-search] 6/31  thr=0.3343  F1=0.73302
[thr-search] 12/31  thr=0.5451  F1=0.72688
[thr-search] 18/31  thr=0.6903  F1=0.66505
[thr-search] 24/31  thr=0.8000  F1=0.55118
[thr-search] 30/31  thr=0.8952  F1=0.38684
[thr-search] 31/31  thr=0.9136  F1=0.35326
[thr-search] best_thr=0.4149  best_f1=0.73815  total=0.7s
[metrics] sampled 2,000,000 rows for AUC/Logloss
[OOF-EMB] BestThr=0.4149 | F1=0.73815 | AUC≈0.81375 | Logloss≈0.50300
Saved OOF: /content/drive/MyDrive/data/instacart/performance3/oof_predictions_instacart_embedding.csv


In [6]:
# @title 6) 테스트(6번째 주문) 예측 & 저장
def predict_with_boosters(df_in: pd.DataFrame, features: List[str], models: List[xgb.Booster], out_col: str):
    X = df_in[features].values.astype(np.float32)
    dmat = xgb.DMatrix(X, feature_names=features)
    preds = np.zeros(len(df_in), dtype=np.float32)
    for bst in models:
        preds += _bst_predict_proba(bst, dmat)
    preds /= max(1, len(models))
    df_in[out_col] = preds
    return preds

predict_with_boosters(df_test, res_EMB["features"], res_EMB["models"], "proba_emb")

# 테스트 평가(임베딩)
test_auc_emb, test_ll_emb = fast_auc_logloss(df_test, target_col, "proba_emb", max_rows=2_000_000)
test_f1_emb = order_level_f1(df_test, order_key, product_col, target_col, "proba_emb", thr_emb)
print(f"[TEST-EMB] Thr={thr_emb:.4f} | F1={test_f1_emb:.5f} | AUC≈{test_auc_emb:.5f} | Logloss≈{test_ll_emb:.5f}")

# 제출 포맷(주문별 products)
def to_pred_string(g: pd.DataFrame, thr: float) -> str:
    items = g.loc[g["proba_emb"] >= thr, product_col].astype(str).tolist()
    return "None" if len(items)==0 else " ".join(items)

sub_series = df_test.groupby(order_key, group_keys=False).apply(lambda g: to_pred_string(g, thr_emb))
sub_df = sub_series.reset_index()
sub_df.columns = [order_key, "products"]
sub_df.to_csv(OUT_ORD_TEST, index=False)

# 상세 저장(행 단위)
df_test[[order_key, member_col, product_col, target_col, "proba_emb"]].to_csv(OUT_DET_TEST, index=False)

print("Saved TEST:")
print(" - detailed:", OUT_DET_TEST)
print(" - orders  :", OUT_ORD_TEST)


/tmp/ipython-input-3101981911.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_in[out_col] = preds


[TEST-EMB] Thr=0.4149 | F1=0.68420 | AUC≈0.75137 | Logloss≈0.57550


/tmp/ipython-input-3101981911.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sub_series = df_test.groupby(order_key, group_keys=False).apply(lambda g: to_pred_string(g, thr_emb))


Saved TEST:
 - detailed: /content/drive/MyDrive/data/instacart/performance3/test_predictions_detailed_embedding.csv
 - orders  : /content/drive/MyDrive/data/instacart/performance3/test_predictions_orders_embedding.csv


In [7]:
# @title 7) 베이스라인 vs 임베딩 — 동일 평가식으로 성능 비교 출력
# GT(6번째 주문) 집합 만들기
df_gt = df_test[[order_key, product_col, target_col]].copy()
df_gt[order_key]   = df_gt[order_key].astype(str)
df_gt[product_col] = df_gt[product_col].astype(str)
true_sets = df_gt.loc[df_gt[target_col]==1].groupby(order_key)[product_col].agg(lambda s: set(s.tolist())).to_dict()

# ----- 임베딩(XGB) 성능: 이미 계산된 변수 사용 -----
res_rows = [{
    "model": "XGB_embedding",
    "macro_F1": float(test_f1_emb),
    "AUC": float(test_auc_emb),
    "Logloss": float(test_ll_emb),
    "orders_used": len(true_sets),     # 전 주문 사용 (확률→임계치)
    "coverage": 1.0,
    "note": "embedding + numeric"
}]

# ----- 베이스라인(XGB) — 주문별 파일로 F1, 상세파일 있으면 AUC/LL -----
base_auc, base_ll, base_f1, base_cov = None, None, None, None

# 1) F1: 주문별 products 파일
if os.path.exists(XGB_ORD_PATH):
    xgb_ord = pd.read_csv(XGB_ORD_PATH)
    id_cand   = [order_key, "order_id","oid","order","test_order_id"]
    prod_cand = ["products","predicted_products","prediction","preds"]
    xgb_id_col  = first_existing(id_cand, xgb_ord.columns.tolist())
    xgb_prodcol = first_existing(prod_cand, xgb_ord.columns.tolist())
    assert xgb_id_col and xgb_prodcol, "XGB 주문별 제출 파일 포맷 확인 필요"

    xgb_pred_sets = {}
    for oid, prod_str in xgb_ord[[xgb_id_col, xgb_prodcol]].itertuples(index=False, name=None):
        xgb_pred_sets[str(oid)] = set(parse_products_to_list(prod_str))

    orders_common = set(xgb_pred_sets.keys()) & set(true_sets.keys())
    f1s = [f1_single(true_sets[oid], xgb_pred_sets.get(oid, set())) for oid in orders_common]
    base_f1 = float(np.mean(f1s)) if len(f1s) else 0.0
    base_cov = len(orders_common) / max(1, len(true_sets))

# 2) AUC/Logloss: 상세 확률 파일
if os.path.exists(XGB_DETAIL_PATH):
    d = pd.read_csv(XGB_DETAIL_PATH)
    # 필요한 열 확인/보강
    need_keys = [order_key, product_col]
    for k in need_keys:
        if k not in d.columns:
            # 없으면 df_test에서 조인해 보강
            d = d.merge(df_test[[order_key, product_col, target_col]], on=[order_key, product_col], how="left")
            break

    # target_col 보장
    if target_col not in d.columns:
        d = d.merge(df_test[[order_key, product_col, target_col]], on=[order_key, product_col], how="left")

    # proba 열 우선순위
    proba_col = first_existing(["proba_ens","proba","pred","p"], d.columns.tolist())
    if (proba_col is not None) and (target_col in d.columns):
        base_auc, base_ll = fast_auc_logloss(d.dropna(subset=[target_col, proba_col]), target_col, proba_col, max_rows=2_000_000)

# 비교 요약 행 추가
res_rows.append({
    "model": "XGB_baseline",
    "macro_F1": base_f1,
    "AUC": base_auc,
    "Logloss": base_ll,
    "orders_used": int(base_cov * len(true_sets)) if base_cov is not None else None,
    "coverage": base_cov,
    "note": "numeric-only baseline (existing files)"
})

# ----- 주문별 F1 비교 테이블 (교집합 기준) -----
# 임베딩 예측 집합 만들기 (방금 저장한 detailed 활용)
emb_pred_sets = {}
det = pd.read_csv(OUT_DET_TEST)
for oid, g in det.groupby(order_key):
    emb_pred_sets[str(oid)] = set(g.loc[g["proba_emb"]>=thr_emb, product_col].astype(str).tolist())

orders_both = list(set(true_sets.keys()) & set(emb_pred_sets.keys()))
if os.path.exists(XGB_ORD_PATH):
    orders_both = list(set(orders_both) & set(pd.read_csv(XGB_ORD_PATH)[first_existing([order_key,"order_id","oid","order","test_order_id"], pd.read_csv(XGB_ORD_PATH).columns.tolist())].astype(str).tolist()))

rows_po = []
for oid in orders_both:
    ts = true_sets[oid]
    es = emb_pred_sets.get(oid, set())
    bs = xgb_pred_sets.get(oid, set()) if 'xgb_pred_sets' in globals() else set()
    rows_po.append({
        "order_id": oid,
        "F1_embedding": f1_single(ts, es),
        "F1_baseline":  f1_single(ts, bs) if len(bs)>0 else np.nan,
        "F1_diff(emb-base)": (f1_single(ts, es) - f1_single(ts, bs)) if len(bs)>0 else np.nan,
        "True_size": len(ts),
        "Emb_pred_size": len(es),
        "Base_pred_size": len(bs) if len(bs)>0 else np.nan
    })
per_order_df = pd.DataFrame(rows_po)
if len(per_order_df):
    per_order_df.sort_values("F1_diff(emb-base)", ascending=False).head(50).to_csv(OUT_EMB_TOP50, index=False)
    per_order_df.sort_values("F1_diff(emb-base)", ascending=True).head(50).to_csv(OUT_BASE_TOP50, index=False)
    per_order_df.to_csv(OUT_PER_ORDER_CMP, index=False)

# ----- 요약 저장 & 출력 -----
summary_df = pd.DataFrame(res_rows)
summary_df.to_csv(OUT_SUMMARY_CSV, index=False)
with open(OUT_SUMMARY_JSON, "w") as f:
    json.dump({"summary": res_rows,
               "notes": {
                   "master": MASTER_PATH,
                   "embedding": EMBED_PATH,
                   "address": ADDRESS_PATH,
                   "baseline_orders": XGB_ORD_PATH if os.path.exists(XGB_ORD_PATH) else None,
                   "baseline_detail": XGB_DETAIL_PATH if os.path.exists(XGB_DETAIL_PATH) else None,
                   "thr_emb_from_oof": float(thr_emb)
               }}, f, indent=2, ensure_ascii=False)

print("\n===== 최종 비교 요약 =====")
print(summary_df)
print("\n저장됨:")
print(" - 비교 요약 CSV:", OUT_SUMMARY_CSV)
print(" - 비교 요약 JSON:", OUT_SUMMARY_JSON)
print(" - 주문별 비교 테이블:", OUT_PER_ORDER_CMP)
print(" - Embedding 우세 TOP50:", OUT_EMB_TOP50)
print(" - Baseline 우세 TOP50:", OUT_BASE_TOP50)



===== 최종 비교 요약 =====
           model  macro_F1       AUC   Logloss  orders_used  coverage  \
0  XGB_embedding  0.684199  0.751375  0.575495       122607       1.0   
1   XGB_baseline  0.751821  0.729420  0.593559       122607       1.0   

                                     note  
0                     embedding + numeric  
1  numeric-only baseline (existing files)  

저장됨:
 - 비교 요약 CSV: /content/drive/MyDrive/data/instacart/performance3/xgb_baseline_vs_embedding_summary.csv
 - 비교 요약 JSON: /content/drive/MyDrive/data/instacart/performance3/xgb_baseline_vs_embedding_summary.json
 - 주문별 비교 테이블: /content/drive/MyDrive/data/instacart/performance3/baseline_vs_embedding_per_order_f1.csv
 - Embedding 우세 TOP50: /content/drive/MyDrive/data/instacart/performance3/embedding_better_top50.csv
 - Baseline 우세 TOP50: /content/drive/MyDrive/data/instacart/performance3/baseline_better_top50.csv
